# Confluence Data Center API サンプル

ローカル Docker で起動した Confluence Data Center に接続して API を試します。

**前提**
- Confluence の **プロフィール → Personal Access Tokens** でトークンを生成済みであること
- `.env` に `CONFLUENCE_DC_URL`, `CONFLUENCE_DC_PAT` を設定済みであること
- `docker compose restart jupyter` で環境変数を反映済みであること

In [10]:
from atlassian import Confluence
from dotenv import load_dotenv
import os
import json

load_dotenv()

# Jupyter コンテナから Confluence への接続
# - Docker ネットワーク内 (Jupyter → Confluence コンテナ): http://confluence:8090
# - ホストから直接アクセスする場合: http://localhost:8090
confluence_url = os.getenv("CONFLUENCE_DC_URL", "http://confluence:8090")

confluence = Confluence(
    url=confluence_url,
    token=os.getenv("CONFLUENCE_DC_PAT"),
)

print(f"接続先: {confluence_url}")

接続先: http://confluence:8090


## スペース一覧の取得

In [11]:
spaces = confluence.get_all_spaces(start=0, limit=25)
print(f"スペース数: {len(spaces.get('results', []))}")
print()
for s in spaces.get("results", []):
    print(f"  [{s['key']}] {s['name']} (type={s['type']})")

スペース数: 2

  [ds] Demonstration Space (type=global)
  [SPC] テスト (type=global)


## スペース内のページ一覧

In [12]:
# 最初のスペースを使用（変更する場合は SPACE_KEY を書き換えてください）
SPACE_KEY = spaces["results"][0]["key"]
print(f"対象スペース: {SPACE_KEY}")
print()

pages = confluence.get_all_pages_from_space(space=SPACE_KEY, start=0, limit=10)
print(f"ページ数: {len(pages)}")
for p in pages:
    print(f"  ID={p['id']}  {p['title']}")

対象スペース: ds

ページ数: 10
  ID=131073  Welcome to Confluence
  ID=131075  Prettify the page with an image (step 4 of 9)
  ID=131077  Share your page with a team member (step 9 of 9)
  ID=131078  Let's edit this page (step 3 of 9)
  ID=131080  Get serious with a table (step 5 of 9)
  ID=131090  A quick look at the editor (step 2 of 9)
  ID=131098  Learn the wonders of autoconvert (step 7 of 9)
  ID=131100  Lay out your page (step 6 of 9)
  ID=131101  Tell people what you think in a comment (step 8 of 9)
  ID=131105  What is Confluence? (step 1 of 9)


## ページ本文の取得

In [13]:
page_id = pages[0]["id"]
page = confluence.get_page_by_id(
    page_id,
    expand="body.storage,version,history,ancestors"
)

print("Title   :", page["title"])
print("Version :", page["version"]["number"])
print("Creator :", page["history"]["createdBy"]["displayName"])
print("Created :", page["history"]["createdDate"])
print()
print("=== Body (storage format, first 1000 chars) ===")
print(page["body"]["storage"]["value"][:1000])

Title   : Welcome to Confluence
Version : 1
Creator : Anonymous
Created : 2020-10-26T15:44:29.341Z

=== Body (storage format, first 1000 chars) ===
<p style="text-align: center;"> </p><h2><ac:image><ri:attachment ri:filename="welcome.png" /></ac:image><br />  <span style="color: rgb(128,128,128);">With Confluence it is easy to create, edit and share content with your team. <br />  Choose a topic below to start learning how.</span></h2><h2><span style="color: rgb(0,0,128);"><br /></span></h2><ol><li><span style="color: rgb(0,0,128);"><ac:link><ri:page ri:content-title="What is Confluence? (step 1 of 9)" /><ac:link-body>What is Confluence?<br /><br /></ac:link-body></ac:link></span></li><li><span style="color: rgb(0,0,128);"><ac:link><ri:page ri:content-title="A quick look at the editor (step 2 of 9)" /><ac:plain-text-link-body><![CDATA[A quick look at the editor]]></ac:plain-text-link-body></ac:link><br /> </span></li><li><span style="color: rgb(0,0,128);"><ac:link><ri:page ri:content-t

## CQL 検索

In [14]:
# CQL (Confluence Query Language) で最近更新されたページを検索
cql_query = f'space="{SPACE_KEY}" AND type=page ORDER BY lastmodified DESC'
results = confluence.cql(cql_query, limit=5)

print(f"CQL: {cql_query}")
print(f"ヒット数: {results.get('totalSize', 0)}")
print()
for r in results.get("results", []):
    title = r.get("title", "(no title)")
    # CQL 結果は _links が content 配下にある場合と url フィールドがある場合がある
    webui = (
        r.get("url")
        or r.get("content", {}).get("_links", {}).get("webui", "")
    )
    url = confluence_url + webui if webui and not webui.startswith("http") else webui
    print(f"  {title}")
    print(f"    {url}")

CQL: space="ds" AND type=page ORDER BY lastmodified DESC
ヒット数: 10

  Share your page with a team member (step 9 of 9)
    http://confluence:8090/spaces/ds/pages/131077/Share+your+page+with+a+team+member+step+9+of+9
  Tell people what you think in a comment (step 8 of 9)
    http://confluence:8090/spaces/ds/pages/131101/Tell+people+what+you+think+in+a+comment+step+8+of+9
  A quick look at the editor (step 2 of 9)
    http://confluence:8090/spaces/ds/pages/131090/A+quick+look+at+the+editor+step+2+of+9
  Lay out your page (step 6 of 9)
    http://confluence:8090/spaces/ds/pages/131100/Lay+out+your+page+step+6+of+9
  What is Confluence? (step 1 of 9)
    http://confluence:8090/spaces/ds/pages/131105/What+is+Confluence+step+1+of+9


## 添付ファイル一覧の取得

In [15]:
attachments = confluence.get_attachments_from_content(page_id=page_id)
att_list = attachments.get("results", [])
print(f"添付ファイル数: {len(att_list)}")
for a in att_list:
    print(f"  {a['title']} ({a['metadata']['mediaType']}, {a['extensions']['fileSize']} bytes)")

添付ファイル数: 1
  welcome.png (image/png, 3070 bytes)
